# 📞 Customer Churn Prediction Portfolio Project

Welcome to the end-to-end portfolio project for **Logistic Regression**. In this notebook, we will tackle a classic business problem: predicting which customers are going to cancel their subscription (churn).

## 1. Business Problem
A telecom company is losing 15% of its customers every month. Customer acquisition costs are high, and it is 5x cheaper to retain an existing customer than to acquire a new one. The company wants a machine learning model to predict **Customer Churn** so they can proactively offer discounts or incentives to high-risk customers.

## 2. Dataset Description
We will generate a synthetic but highly realistic Telecom Churn dataset for this project to simulate typical industry data.

**Features:**
- `MonthlyCharges`: The amount the customer pays per month.
- `Tenure`: How many months the customer has been with the company.
- `NumSupportTickets`: How many times they called support in the last month.
- `ContractType`: Month-to-month, 1-year, or 2-year.
- **Target:** `Churn` (1 if they left, 0 if they stayed).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data
n_samples = 5000

tenure = np.random.randint(1, 72, n_samples)
monthly_charges = np.random.uniform(20, 120, n_samples)
support_tickets = np.random.poisson(lam=1.5, size=n_samples)
contract_type = np.random.choice(['Month-to-month', 'One year', 'Two year'], size=n_samples, p=[0.5, 0.3, 0.2])

# Logic for churn: High tickets, low tenure, high charges, and month-to-month contracts increase churn probability
z = -2.0 + 0.8 * support_tickets - 0.05 * tenure + 0.02 * monthly_charges + \
    np.where(contract_type == 'Month-to-month', 1.5, 0) - \
    np.where(contract_type == 'Two year', 2.0, 0)
    
prob_churn = 1 / (1 + np.exp(-z))
churn = (np.random.rand(n_samples) < prob_churn).astype(int)

df = pd.DataFrame({
    'Tenure': tenure,
    'MonthlyCharges': monthly_charges,
    'NumSupportTickets': support_tickets,
    'ContractType': contract_type,
    'Churn': churn
})

df.head()

## 3. Exploratory Data Analysis (EDA)
Let's visually understand what factors drive churn.

In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
sns.boxplot(x='Churn', y='Tenure', data=df)
plt.title('Tenure vs Churn')

plt.subplot(1, 3, 2)
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges vs Churn')

plt.subplot(1, 3, 3)
sns.countplot(x='ContractType', hue='Churn', data=df)
plt.title('Contract Type vs Churn')

plt.tight_layout()
plt.show()

## 4. Feature Engineering & Preprocessing
We need to One-Hot Encode our categorical variables and Standardize our numerical ones for Logistic Regression.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['ContractType'], drop_first=True)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data shape:", X_train_scaled.shape)

## 5. Model Training & 6. Hyperparameter Tuning
We will use `GridSearchCV` to find the best L2 regularization penalty (`C`).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(LogisticRegression(solver='lbfgs', max_iter=1000), param_grid, cv=5, scoring='roc_auc')
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

## 7. Evaluation & 8. Error Analysis
Because churn datasets are typically imbalanced, we rely on ROC-AUC, Precision, and Recall rather than pure accuracy.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

y_pred = best_model.predict(X_test_scaled)
y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix Visualization
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 9. Visualizations: ROC Curve
The ROC curve plots the True Positive Rate against the False Positive Rate.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, color='orange', label='ROC (AUC = %0.2f)' % roc_auc_score(y_test, y_prob))
plt.plot([0, 1], [0, 1], color='darkblue', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.show()

## 10. Business Insights
Let's look at the weights assigned by our model to understand *why* people leave.

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.coef_[0]
})
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x='Importance', y='Feature', data=feature_importance, orient='h')
plt.title("Logistic Regression Coefficients (Business Drivers)")
plt.show()

print("Key Business Insights:")
print("- **Support Tickets** are the strongest driver of churn. If a user calls support heavily, they are frustrated.")
print("- **Monthly Charges** slightly increase churn.")
print("- **Tenure** and long-term contracts strongly REDUCE churn. We should incentivize customers to sign 2-year contracts.")

## 11. Future Improvements
- **Decision Threshold Optimization:** The default threshold is 0.5. We could lower it to 0.3 to catch more potential churners at the cost of offering discounts to some people who would have stayed anyway.
- **Non-Linear Models:** A Random Forest or XGBoost model might capture non-linear interactions (e.g., high tenure + high charges behaves differently than low tenure + high charges).